Project Number 1: Churn prediction

first step, Download the dependencies

In [362]:
!pip install kagglehub
!pip install pandas
!pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


import the kaggle dataset with kagglehub

In [363]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("blastchar/telco-customer-churn")

print("Path to dataset files:", path)

Path to dataset files: /home/jorgeacr/.cache/kagglehub/datasets/blastchar/telco-customer-churn/versions/1


List the name of the csv file

In [364]:
import os

print(os.listdir(path))

['WA_Fn-UseC_-Telco-Customer-Churn.csv']


Load the csv and do EDA looking for null values

In [365]:
import pandas as pd

df = pd.read_csv(path + '/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [366]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Seeing the info about the csv we can se that TotalCharges is a str, so we might fix it to check for NaN values

In [367]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

We convert TotalCharges to numeric, and make no numeric values turn NaN

In [368]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors = 'coerce')

Check for NaN values

In [369]:
df.isnull().sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

Drop Null values from the csv since they're only 11 cases

In [370]:
df.dropna(subset=['TotalCharges'])

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,Yes


We check the proportion in de dataset between clases

In [383]:
df['Churn'].value_counts(normalize=True)

Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

Now, start creating the pipeline

encoding str columns

In [372]:
df['gender'] = df['gender'].map({'Male':1,'Female':0})

In [373]:
df = df.drop("customerID", axis=1, errors="ignore")
y = df["Churn"]
X = df.drop("Churn", axis=1)
X = pd.get_dummies(X, drop_first=True)
X = X.fillna(X.median())

In [374]:
y = y.map({"No": 0, "Yes": 1})

Train test split 

In [375]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [376]:
print(X.shape)
print(y.shape)
print(y.head())

(7043, 30)
(7043,)
0    0
1    0
2    1
3    0
4    1
Name: Churn, dtype: int64


Create the model and train it with the data

In [377]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, class_weight= 'balanced')

In [378]:
model.fit(X_train,y_train)

/home/jorgeacr/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

Program a thredshold to make more accurate predictions for churn

In [379]:
y_proba = model.predict_proba(X_test)[:, 1]

In [380]:
y_pred = (y_proba > 0.3).astype(int)

Print the metrics of the model Preccision ~ Recall 

In [381]:
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.56      0.70      1035
           1       0.43      0.93      0.59       374

    accuracy                           0.66      1409
   macro avg       0.69      0.74      0.65      1409
weighted avg       0.82      0.66      0.67      1409



The confusion Matrix

In [382]:
print(confusion_matrix(y_test, y_pred))

[[578 457]
 [ 27 347]]


Notes:

## Project Overview

This project explores a customer churn dataset from a telecommunications company, containing over 7,000 records. The dataset includes user-level information such as demographic attributes (e.g., gender), subscribed services, billing details, payment methods, and a binary target variable indicating whether the customer churned.

A key characteristic of the dataset is the class imbalance, with approximately a 70/30 split between non-churn and churn cases. This imbalance significantly influenced both model behavior and evaluation strategy throughout the project.

---

## Data Challenges and Preprocessing

The dataset required substantial preprocessing before modeling:

* Several features were categorical (string-based) and required encoding into numerical representations.
* Careful handling of encoding was necessary to avoid data leakage and ensure compatibility with downstream models.
* Missing values and data type inconsistencies (e.g., numeric fields stored as strings) required additional cleaning steps.

During this phase, multiple issues arose, including incorrect transformations, pipeline inconsistencies, and common implementation errors (e.g., missing imports, syntax mistakes). These challenges were instrumental in developing a deeper understanding of data preprocessing workflows.

---

## Modeling Approach

A baseline Logistic Regression model was trained after preprocessing. Initial performance was suboptimal, particularly in detecting churn cases, largely due to class imbalance and default decision thresholds.

To address this, two key strategies were applied:

* Class imbalance handling (e.g., weighting)
* Decision threshold tuning

By lowering the classification threshold to 0.3, the model significantly improved its ability to detect churn cases.

---

## Results and Trade-offs

After threshold adjustment, the model achieved:

* High recall for churn (~93%)
* Lower precision (~43%), indicating a high number of false positives

This reflects a deliberate trade-off:

> The model prioritizes capturing as many churn cases as possible, at the expense of increased false alarms.

---

## Business Interpretation

This model would be viable in scenarios where the cost of losing a customer is significantly higher than the cost of intervention (e.g., marketing campaigns, discounts, retention offers).

In such contexts:

* False positives are acceptable (unnecessary outreach)
* False negatives are costly (lost customers)

Thus, the model aligns with a **recall-oriented business strategy**.

---

## Reflections

As a first machine learning project, this experience involved a steep learning curve and frequent errors, ranging from implementation bugs to conceptual misunderstandings.

However, these challenges were essential in developing practical intuition around:

* Data preprocessing pipelines
* Handling class imbalance
* Model evaluation beyond accuracy
* Threshold-based decision making

This project marked a transition from theoretical understanding to applied machine learning practice. Future work will focus on improving model performance through feature engineering, more advanced models, and robust pipeline design.
